### Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt
import tifffile
from scipy.signal import find_peaks
from scipy.stats import sem

### Parameters and Directory
##### Must be adapted according to experiment

In [ ]:
### PARAMETERS
folder = r"......."
filetype = ".tif"
framerate=5#inHz
n_patches = 30
delta = 5  # lag in frames
window_before = 50
window_after = 100
expected_len = window_before + window_after + 1


### Main Script

In [ ]:
### OUTPUT SUBFOLDER
analysis_folder = os.path.join(folder, "analysis")
os.makedirs(analysis_folder, exist_ok=True)

### ANALYSIS FUNCTIONS

def getCorrelations(movie, n, lag):
    time, x, y = movie.shape
    patch_x, patch_y = x // n, y // n
    otsu_thresh = threshold_otsu(movie)
    correlations = np.zeros((n*n, time - lag))
    for patch_idx in range(n*n):
        i = patch_idx // n
        j = patch_idx % n
        for t in range(time - lag):
            patch1 = movie[t, i*patch_x:(i+1)*patch_x, j*patch_y:(j+1)*patch_y]
            patch2 = movie[t+lag, i*patch_x:(i+1)*patch_x, j*patch_y:(j+1)*patch_y]
            mask = (patch1 > otsu_thresh) & (patch2 > otsu_thresh)
            p1 = patch1[mask]
            p2 = patch2[mask]
            if p1.size < 2:
                corr = np.nan
            else:
                corr = np.corrcoef(p1, p2)[0, 1]
            correlations[patch_idx, t] = corr
    return correlations


def norm01(arr):
    return (arr - np.nanmin(arr)) / (np.nanmax(arr) - np.nanmin(arr))

### BATCH PROCESSING

all_I_snippets = []
all_corrs_snippets = []
names = []

for fname in sorted(os.listdir(folder)):
    if not fname.endswith(filetype):
        continue
    
    print(f"Processing {fname}")
    try:
        # Load image
        image_path = os.path.join(folder, fname)
        imageStack = tifffile.imread(image_path)
        calciumImage = imageStack[:, 0, :, :]
        refImage = imageStack[:, 1, :, :]

        # Extract tables
        corrs = getCorrelations(refImage, n_patches, delta)
        corrs_table = pd.DataFrame(corrs).T
        I_table = pd.Series(calciumImage.mean(axis=1).mean(axis=1))

        # --- SAVE PER-FILE TABLES ---
        basename = os.path.splitext(fname)[0]
        I_table.to_csv(os.path.join(analysis_folder, f"{basename}_I_table.csv"), index=False)
        corrs_table.to_csv(os.path.join(analysis_folder, f"{basename}_corrs_table.csv"), index=False)

        # Find peaks
        peaks, _ = find_peaks(I_table, distance=50)
        # Get windows
        for peak in peaks:
            
            start = peak - window_before
            end = peak + window_after + 1
            if start < 0 or end > len(I_table):
                continue
            I_window = I_table.iloc[start:end].values
            corrs_window = corrs_table.iloc[start:end].mean(axis=1).values
            # Check window length
            if len(I_window) != expected_len or len(corrs_window) != expected_len:
                continue
            all_I_snippets.append(I_window)
            all_corrs_snippets.append(corrs_window)
            names += [f"{fname}_{peak}"]
    except Exception as e:
        print(f"Error processing {fname}: {e}")

print(f"Number of valid Ca peaks/windows: {len(all_I_snippets)}")



### Generate Summary Files

In [ ]:

# --- Normalize each snippet individually ---
I_snippets = np.array([norm01(snip) for snip in all_I_snippets])
corrs_snippets = np.array([norm01(snip) for snip in all_corrs_snippets])

I_avg_norm = I_snippets.mean(axis=0)
corrs_avg_norm = corrs_snippets.mean(axis=0)
I_sem_norm = sem(I_snippets, axis=0, nan_policy='omit')
corrs_sem_norm = sem(corrs_snippets, axis=0, nan_policy='omit')

I_summary = pd.DataFrame(I_snippets).T
I_summary.columns = names

Corr_summary = pd.DataFrame(corrs_snippets).T
Corr_summary.columns = names

# --- SAVE SUMMARY CSVs ---
I_summary.to_csv(os.path.join(analysis_folder, "I_summary.csv"))
Corr_summary.to_csv(os.path.join(analysis_folder, "Corr_summary.csv"))


### Plot the Data

In [ ]:
frames = np.arange(-window_before, window_after + 1)
time=frames / framerate
fig, ax1 = plt.subplots(figsize=(8, 4))

color1 = 'tab:blue'
color2 = 'tab:orange'

# Ca2+ (avg, 0–1 norm) on left y-axis
ax1.set_xlabel('Time [s]', fontsize=13)
ax1.set_ylabel('Ca$^{2+}$ Signal [a.u.]', color=color1, fontsize=13, labelpad=12)
line1 = ax1.plot(time, I_avg_norm, color=color1, label='Ca$^{2+}$ (avg, 0–1 norm)')
ax1.fill_between(time, I_avg_norm - I_sem_norm, I_avg_norm + I_sem_norm, alpha=0.3, color=color1)
ax1.tick_params(axis='y', labelcolor=color1, labelsize=12)
ax1.tick_params(axis='x', labelsize=12)

# Correlation (1-corr, 0–1 norm) on right y-axis
ax2 = ax1.twinx()
ax2.set_ylabel('Relative Movement [a.u.]', color=color2, fontsize=13, labelpad=12)
line2 = ax2.plot(time, 1-corrs_avg_norm, color=color2, label='1 - Correlation (avg, 0–1 norm)')
ax2.fill_between(time, 1-corrs_avg_norm - corrs_sem_norm, 1-corrs_avg_norm + corrs_sem_norm, alpha=0.3, color=color2)
ax2.tick_params(axis='y', labelcolor=color2, labelsize=12)
fig.tight_layout()

plot_path = os.path.join(analysis_folder, "average_ca_corr_plot_2yaxes.png")
plt.savefig(plot_path, dpi=300)
plt.show()
